# Chapter 9: The PyMC Ecosystem — MCMC in Practice

## quap vs MCMC

All previous notebooks used **quap** (quadratic approximation): find the MAP, assume the posterior is Gaussian around it. It's fast and works well when the posterior really is bell-shaped.

**When quap breaks down**:
- Skewed or multimodal posteriors
- Hierarchical models with funnel geometry
- Discrete parameters
- Any model where the Gaussian assumption is obviously wrong

**MCMC** (specifically **NUTS** — No-U-Turn Sampler, the default in PyMC) makes no shape assumptions. It explores the posterior by simulating a Hamiltonian system, drawing correlated samples that, in aggregate, approximate the true posterior.

## This Notebook

We already know the ruggedness/Africa interaction model (m3) well from `interactions_ruggedness.ipynb`. Now we re-implement all three models in **PyMC** and learn the full modern workflow:

| Step | Tool |
|------|------|
| Model definition | `pm.Model()`, `pm.Normal`, `pm.Exponential` |
| Prior predictive | `pm.sample_prior_predictive()` |
| Sampling | `pm.sample()` (NUTS/HMC) |
| Diagnostics | `az.summary()`, `az.plot_trace()`, R-hat, ESS |
| Posteriors | `az.plot_posterior()`, `az.plot_forest()`, `az.plot_pair()` |
| Posterior predictive | `pm.sample_posterior_predictive()` |
| Model comparison | `az.waic()`, `az.compare()` |
| vs quap | Side-by-side comparison |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
from scipy import stats
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))
from src.quap import quap

plt.style.use('default')
%matplotlib inline

print(f'PyMC  version: {pm.__version__}')
print(f'ArviZ version: {az.__version__}')

## Load Data (same as interactions notebook)

In [ ]:
url = 'https://raw.githubusercontent.com/rmcelreath/rethinking/master/data/rugged.csv'
raw = pd.read_csv(url, sep=';')
df  = raw[['country', 'cont_africa', 'rugged', 'rgdppc_2000']].dropna().copy().reset_index(drop=True)

df['log_gdp']     = np.log(df['rgdppc_2000'])
df['log_gdp_std'] = df['log_gdp'] / df['log_gdp'].mean()
df['rugged_std']  = df['rugged'] / df['rugged'].max()
df['cid']         = df['cont_africa'].astype(int)   # 0=non-Africa, 1=Africa

y    = df['log_gdp_std'].values.astype(float)
r    = df['rugged_std'].values.astype(float)
cid  = df['cid'].values.astype(int)
r_bar = r.mean()
r_c  = r - r_bar   # centred ruggedness

print(f'N = {len(y)},  Africa = {cid.sum()},  non-Africa = {(1-cid).sum()}')
print(f'r_bar = {r_bar:.3f}')

## Define the Three Models in PyMC

### PyMC model anatomy

```python
with pm.Model() as model:
    # 1. Priors — random variables
    alpha = pm.Normal('alpha', mu=1, sigma=0.1)

    # 2. Deterministic transform (optional)
    mu = alpha + beta * x

    # 3. Likelihood — observed node
    obs = pm.Normal('obs', mu=mu, sigma=sigma, observed=y)

# Everything inside the `with` block is part of the model graph.
# Outside it, you sample:
with model:
    idata = pm.sample()
```

`coords` + `dims` are optional but make ArviZ plots label axes correctly.

In [ ]:
coords = {'continent': ['non_africa', 'africa']}

# ── m1: pooled — no continent information ─────────────────────────────────────
with pm.Model() as model_m1:
    alpha = pm.Normal('alpha', mu=1,   sigma=0.1)
    beta  = pm.Normal('beta',  mu=0,   sigma=0.3)
    sigma = pm.Exponential('sigma', lam=1)
    mu    = alpha + beta * r_c
    obs   = pm.Normal('obs', mu=mu, sigma=sigma, observed=y)

# ── m2: separate intercepts, shared slope ─────────────────────────────────────
with pm.Model(coords=coords) as model_m2:
    alpha = pm.Normal('alpha', mu=1,   sigma=0.1, dims='continent')
    beta  = pm.Normal('beta',  mu=0,   sigma=0.3)
    sigma = pm.Exponential('sigma', lam=1)
    mu    = alpha[cid] + beta * r_c
    obs   = pm.Normal('obs', mu=mu, sigma=sigma, observed=y)

# ── m3: full interaction — separate intercepts AND slopes ─────────────────────
with pm.Model(coords=coords) as model_m3:
    alpha = pm.Normal('alpha', mu=1,   sigma=0.1, dims='continent')
    beta  = pm.Normal('beta',  mu=0,   sigma=0.3, dims='continent')
    sigma = pm.Exponential('sigma', lam=1)
    mu    = alpha[cid] + beta[cid] * r_c
    obs   = pm.Normal('obs', mu=mu, sigma=sigma, observed=y)

print('Models defined. Printing m3 model graph:')
print(model_m3)

## Prior Predictive Check

`pm.sample_prior_predictive()` draws from the prior and propagates through the model — **without looking at the data**. This lets us check that our priors produce plausible outcomes before fitting.

We check m3 here (the most complex model, so most important to validate).

In [ ]:
with model_m3:
    prior_pc = pm.sample_prior_predictive(samples=200, random_seed=42)

# prior_pc is an InferenceData object; prior samples live in prior_pc.prior
# prior predictive (the obs distribution) lives in prior_pc.prior_predictive
print('Prior predictive object structure:')
print(prior_pc)
print()
print('Alpha prior samples shape:', prior_pc.prior['alpha'].shape)  # (chain, draw, continent)
print('Beta  prior samples shape:', prior_pc.prior['beta'].shape)

In [ ]:
# Plot: 50 prior regression lines for Africa and non-Africa
alpha_prior = prior_pc.prior['alpha'].values.reshape(-1, 2)   # (draws, 2)
beta_prior  = prior_pc.prior['beta'].values.reshape(-1, 2)    # (draws, 2)

r_seq = np.linspace(0, 1, 100)
n_lines = 50

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, (cid_val, color, label) in zip(axes, [
        (0, 'steelblue', 'Non-Africa'),
        (1, 'tomato',    'Africa')]):
    subset = df[df['cid'] == cid_val]
    ax.scatter(subset['rugged_std'], subset['log_gdp_std'],
               s=20, alpha=0.3, color=color, zorder=2)
    for i in range(n_lines):
        a = alpha_prior[i, cid_val]
        b = beta_prior[i,  cid_val]
        ax.plot(r_seq, a + b * (r_seq - r_bar), color=color, alpha=0.2, lw=1)
    ax.set_xlabel('Ruggedness (std)', fontsize=11)
    ax.set_ylabel('log GDP (std)',    fontsize=11)
    ax.set_title(f'{label} — prior predictive lines', fontsize=12, fontweight='bold')
    ax.set_ylim(0.5, 1.5)
    ax.grid(True, alpha=0.3)

plt.suptitle('m3 Prior Predictive Check: α~N(1,0.1), β~N(0,0.3)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Sample with NUTS

`pm.sample()` runs **4 chains** (default) of **NUTS** in parallel. Each chain:
1. Warms up (tunes step size and mass matrix) — these samples are discarded
2. Draws posterior samples

The result is an **InferenceData** object (ArviZ standard format) containing the posterior, log-likelihood, and metadata.

In [ ]:
SAMPLE_KWARGS = dict(draws=1000, tune=1000, chains=4,
                     target_accept=0.9, random_seed=42,
                     progressbar=True)

with model_m1:
    idata_m1 = pm.sample(**SAMPLE_KWARGS)
    pm.compute_log_likelihood(idata_m1)

print('m1 done')

In [ ]:
with model_m2:
    idata_m2 = pm.sample(**SAMPLE_KWARGS)
    pm.compute_log_likelihood(idata_m2)

print('m2 done')

In [ ]:
with model_m3:
    idata_m3 = pm.sample(**SAMPLE_KWARGS)
    pm.compute_log_likelihood(idata_m3)

print('m3 done')

## Diagnostics: Did the Chains Mix?

Before trusting any parameter estimates, check that sampling went well.

### Key diagnostics

| Metric | What it checks | Good value |
|--------|---------------|------------|
| **R-hat** (`r_hat`) | Chains converged to same distribution | ≈ 1.0 (< 1.01 ideal) |
| **ESS bulk** | Effective sample size for bulk of posterior | > 100 per chain |
| **ESS tail** | Effective sample size for tails (quantiles) | > 100 per chain |
| **MCSE** | Monte Carlo standard error of the mean | Small relative to SD |
| **Divergences** | HMC trajectory failures — sign of geometry problems | 0 |

The **trace plot** shows chain paths over time — should look like "fuzzy caterpillars" with no trends or stuck regions.

In [ ]:
# Summary table (m3)
summary_m3 = az.summary(idata_m3, hdi_prob=0.89)
print('m3 posterior summary (89% HDI):')
print('=' * 75)
print(summary_m3.to_string())
print('=' * 75)
print()

# Flag any diagnostics issues
if (summary_m3['r_hat'] > 1.01).any():
    print('⚠  Some R-hat > 1.01 — chains may not have converged')
else:
    print('✓ All R-hat ≤ 1.01 — chains converged')

if (summary_m3['ess_bulk'] < 400).any():
    print('⚠  Some ESS bulk < 400 — consider more draws')
else:
    print('✓ ESS bulk all > 400 — sufficient effective samples')

n_div = idata_m3.sample_stats['diverging'].values.sum()
print(f'✓ Divergences: {n_div}' if n_div == 0 else f'⚠  Divergences: {n_div} — geometry issues')

In [ ]:
# Trace plots — "fuzzy caterpillars" = good mixing
az.plot_trace(idata_m3, figsize=(12, 8))
plt.suptitle('m3 Trace Plots — Left: posterior distribution, Right: chain paths',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Posterior Visualisation

ArviZ has a rich set of posterior visualisation tools. We use three:

- **`plot_posterior`**: histogram + HDI for each parameter
- **`plot_forest`**: compact comparison of multiple parameters or models
- **`plot_pair`**: pairwise joint posteriors — reveals parameter correlations

In [ ]:
# plot_posterior: one panel per parameter, 89% HDI
az.plot_posterior(idata_m3, hdi_prob=0.89,
                  var_names=['alpha', 'beta', 'sigma'],
                  figsize=(14, 4))
plt.suptitle('m3 Posterior Distributions (89% HDI)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# plot_forest: compact overview, great for comparing parameter subsets
az.plot_forest(idata_m3, var_names=['alpha', 'beta'],
               hdi_prob=0.89, combined=True,
               figsize=(9, 4))
plt.axvline(0, color='red', ls='--', lw=1.2, alpha=0.6)
plt.title('m3 Forest Plot — intercepts and slopes by continent (89% HDI)',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# plot_pair: joint posteriors — reveals correlations between parameters
# Useful for diagnosing posterior geometry
az.plot_pair(idata_m3,
             var_names=['alpha', 'beta'],
             divergences=True,
             figsize=(10, 8))
plt.suptitle('m3 Pairwise Joint Posteriors\nDivergences shown in orange (if any)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Posterior Predictive Check

`pm.sample_posterior_predictive()` draws new observations from the fitted model — plugging in posterior samples of parameters. This checks whether the *model's predictions look like the actual data*.

A good model: predictive distribution should contain most observed data points.

In [ ]:
with model_m3:
    ppc = pm.sample_posterior_predictive(idata_m3, random_seed=42)

# ppc.posterior_predictive['obs'] has shape (chains, draws, N_obs)
obs_pred = ppc.posterior_predictive['obs'].values.reshape(-1, len(y))  # (S, N)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: predicted vs observed (scatter)
ax = axes[0]
pred_mean = obs_pred.mean(axis=0)
pred_lo   = np.percentile(obs_pred, 5.5,  axis=0)
pred_hi   = np.percentile(obs_pred, 94.5, axis=0)
ax.errorbar(y, pred_mean,
            yerr=[pred_mean - pred_lo, pred_hi - pred_mean],
            fmt='o', ms=4, alpha=0.5, color='steelblue',
            ecolor='steelblue', elinewidth=0.8, capsize=2)
lo, hi = min(y.min(), pred_mean.min()), max(y.max(), pred_mean.max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5, alpha=0.5, label='perfect prediction')
ax.set_xlabel('Observed log GDP (std)', fontsize=12)
ax.set_ylabel('Predicted mean ± 89% PI', fontsize=12)
ax.set_title('Predicted vs Observed', fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 2: distribution of predictive vs observed
ax = axes[1]
az.plot_ppc(ppc, observed=True, num_pp_samples=200,
            var_names=['obs'], ax=ax)
ax.set_title('Posterior Predictive Distribution\nvs Observed', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('m3 Posterior Predictive Check', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Slope Reversal — Posterior Regression Lines

Use the MCMC posterior samples directly to reconstruct the slope-reversal plot.

In [ ]:
# Extract posterior samples from idata
post = idata_m3.posterior
alpha_s = post['alpha'].values.reshape(-1, 2)   # (S, 2)
beta_s  = post['beta'].values.reshape(-1, 2)    # (S, 2)

r_seq = np.linspace(0, 1, 100)
africa     = df[df['cid'] == 1]
non_africa = df[df['cid'] == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, (subset, cid_val, color, label) in zip(axes, [
        (non_africa, 0, 'steelblue', 'Non-Africa'),
        (africa,     1, 'tomato',    'Africa')]):
    ax.scatter(subset['rugged_std'], subset['log_gdp_std'],
               s=40, alpha=0.6, color=color, zorder=3)

    mu_mat = alpha_s[:, cid_val, None] + beta_s[:, cid_val, None] * (r_seq[None, :] - r_bar)
    mu_mean = mu_mat.mean(0)
    mu_lo   = np.percentile(mu_mat, 5.5,  axis=0)
    mu_hi   = np.percentile(mu_mat, 94.5, axis=0)

    ax.fill_between(r_seq, mu_lo, mu_hi, alpha=0.2, color=color)
    ax.plot(r_seq, mu_mean, color=color, lw=2.5, label='m3 posterior mean (89% CI)')

    slope_mean = beta_s[:, cid_val].mean()
    ax.set_xlabel('Ruggedness (std)', fontsize=12)
    ax.set_ylabel('log GDP (std)',    fontsize=12)
    ax.set_title(f'{label}\nβ = {slope_mean:+.3f}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('m3 Posterior Regression Lines — Slope Reversal\n(MCMC posterior samples)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## quap vs MCMC — Side-by-Side Comparison

For this model, the posterior is close to Gaussian, so quap and MCMC should agree well. When they disagree, MCMC is more trustworthy.

In [ ]:
# Re-fit m3 with quap for comparison
def fit_m3_quap(y, r_c, cid):
    def nlp(params):
        a0, a1, b0, b1, log_sigma = params
        sigma = np.exp(log_sigma)
        alpha = np.array([a0, a1]); beta = np.array([b0, b1])
        mu = alpha[cid] + beta[cid] * r_c
        ll = np.sum(stats.norm.logpdf(y, mu, sigma))
        lp = (np.sum(stats.norm.logpdf([a0, a1], 1, 0.1)) +
              np.sum(stats.norm.logpdf([b0, b1], 0, 0.3)) +
              stats.expon.logpdf(sigma, scale=1))
        return -(ll + lp + log_sigma)
    m = quap(nlp, [1.05, 0.9, -0.1, 0.1, np.log(0.1)],
             ['alpha_0', 'alpha_1', 'beta_0', 'beta_1', 'log_sigma'])
    m.transform_param('log_sigma', 'sigma', np.exp)
    return m

m3_quap = fit_m3_quap(y, r_c, cid)

# Build comparison table
quap_coef = m3_quap.coef()
quap_std  = {k: v for k, v in zip(m3_quap.param_names,
             np.sqrt(np.diag(m3_quap.cov())))}

mcmc_summary = az.summary(idata_m3, hdi_prob=0.89, round_to=4)

param_map = {
    'alpha[non_africa]': ('alpha_0', 'α non-Africa'),
    'alpha[africa]':     ('alpha_1', 'α Africa'),
    'beta[non_africa]':  ('beta_0',  'β non-Africa'),
    'beta[africa]':      ('beta_1',  'β Africa'),
    'sigma':             ('sigma',   'σ'),
}

rows = []
for mcmc_key, (quap_key, label) in param_map.items():
    mcmc_row = mcmc_summary.loc[mcmc_key] if mcmc_key in mcmc_summary.index else None
    if mcmc_row is not None:
        rows.append({
            'Parameter':   label,
            'quap mean':   round(quap_coef.get(quap_key, np.nan), 4),
            'quap SD':     round(quap_std.get(quap_key, np.nan), 4),
            'MCMC mean':   round(float(mcmc_row['mean']), 4),
            'MCMC SD':     round(float(mcmc_row['sd']),   4),
            'R-hat':       round(float(mcmc_row['r_hat']), 3),
        })

df_cmp = pd.DataFrame(rows)
print('quap vs MCMC comparison (m3):')
print('=' * 70)
print(df_cmp.to_string(index=False))
print('=' * 70)
print()
print('Takeaway: for this model the posteriors are nearly Gaussian,')
print('so quap and MCMC agree closely. quap is ~100× faster here.')

## Model Comparison with WAIC (ArviZ)

`az.compare()` takes a dict of InferenceData objects and returns a ranked table with WAIC (or LOO) scores, standard errors, and weights. This replaces the manual WAIC we wrote in Chapter 7.

In [ ]:
comparison = az.compare(
    {'m1: pooled':              idata_m1,
     'm2: continent intercepts': idata_m2,
     'm3: full interaction':    idata_m3},
    ic='waic',
    scale='deviance'   # lower = better (matches McElreath convention)
)

print('ArviZ WAIC model comparison:')
print('=' * 80)
print(comparison.to_string())
print('=' * 80)
print()
print('rank=0 is best. dse = SE of difference from best model.')
print('weight = approximate posterior model probability.')

In [ ]:
# ArviZ comparison plot
az.plot_compare(comparison, figsize=(9, 4))
plt.title('WAIC Model Comparison (ArviZ)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Posterior Contrast: Slope Difference (MCMC version)

Posterior contrasts from MCMC samples — same idea as before, but now the samples come from NUTS rather than the Gaussian approximation.

In [ ]:
beta_nonaf = post['beta'].sel(continent='non_africa').values.flatten()
beta_af    = post['beta'].sel(continent='africa').values.flatten()
delta_beta = beta_af - beta_nonaf

alpha_nonaf = post['alpha'].sel(continent='non_africa').values.flatten()
alpha_af    = post['alpha'].sel(continent='africa').values.flatten()
delta_alpha = alpha_nonaf - alpha_af   # non-Africa richer → positive

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (contrast, title, color) in zip(axes, [
        (delta_beta,  'β_Africa − β_non-Africa\n(slope contrast)',     'darkorchid'),
        (delta_alpha, 'α_non-Africa − α_Africa\n(intercept contrast)', 'steelblue')]):
    az.plot_posterior({'contrast': contrast}, hdi_prob=0.89,
                      ref_val=0, ax=ax, color=color)
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.suptitle('Posterior Contrasts from MCMC', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

ci_b = np.percentile(delta_beta,  [5.5, 94.5])
ci_a = np.percentile(delta_alpha, [5.5, 94.5])
print(f'Slope contrast    β_Af − β_nonAf:  mean={delta_beta.mean():+.3f},  89% CI=[{ci_b[0]:+.3f}, {ci_b[1]:+.3f}],  P>0={( delta_beta>0).mean():.1%}')
print(f'Intercept contrast α_nonAf − α_Af: mean={delta_alpha.mean():+.3f},  89% CI=[{ci_a[0]:+.3f}, {ci_a[1]:+.3f}],  P>0={(delta_alpha>0).mean():.1%}')

## Key Insights: The PyMC Ecosystem

### 1. The InferenceData object is the currency
Everything flows through `InferenceData` (ArviZ's standard format):
- `idata.posterior` — samples from the posterior
- `idata.prior` — prior predictive samples
- `idata.posterior_predictive` — posterior predictive samples
- `idata.log_likelihood` — pointwise log-likelihoods (needed for WAIC/LOO)
- `idata.sample_stats` — divergences, step sizes, acceptance rates, energy

### 2. Always check diagnostics before trusting estimates
```
R-hat ≈ 1.00   →  chains converged
ESS > 400      →  enough effective samples
Divergences = 0 → no geometry problems
Fuzzy trace    →  good mixing
```
If any of these fail, the estimates are unreliable — fix the model before interpreting.

### 3. quap is fine for simple models; PyMC for everything else
For models with roughly Gaussian posteriors (linear regression, GLMs with large N), quap is ~100x faster and gives the same answer. Use PyMC when:
- The model is hierarchical (Chapter 13+)
- Posteriors are skewed or multimodal
- You need divergence checks or ESS diagnostics
- You want the full ArviZ workflow (`az.compare`, `plot_ppc`, etc.)

### 4. `az.compare()` replaces manual WAIC
As long as `pm.compute_log_likelihood()` has been called, one line gives you WAIC, LOO, SE of difference, and stacking weights across all models.

### 5. `dims` and `coords` are worth specifying
Labelling array dimensions (e.g. `dims='continent'`, `coords={'continent': ['non_africa', 'africa']}`) makes ArviZ plots readable without manual tweaking.

---

*Chapter 9 — Statistical Rethinking (McElreath, 2nd ed.)*